In [1]:
n_jobs= 60
#libraries
import pandas as pd
import numpy as np
import os
import sys
from glob import glob
import joblib
import warnings
from datetime import date, datetime
import copy

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from scipy.stats import pearsonr
import scipy.stats as st

from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning

import pickle

from sklearn.compose import TransformedTargetRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import PowerTransformer

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#path to dir
path='/scratch2/alinat/project/PD-EEG-ANON_eegOnly/MLtables180'

In [3]:
#load lvl1
#load prepared data
fnm = '/MLout/MLlvl1_1targ_6algs_ZnoLG.pkl'

# Load from file
with open(path+fnm, 'rb') as file:
    MLlvl1 = pickle.load(file)

In [4]:
#load lvl2
#save/load prepared data
fnm = '/MLout/MLlvl2_1targ_6algs_ZnoLG_full.pkl'


# Load from file
with open(path+fnm, 'rb') as file:
    MLlvl2 = pickle.load(file)

In [5]:
#load lvl3 flat
#save/load prepared data
fnm = '/MLout/MLlvl3_1targ_4algs_FLAT_ZnoLG_full.pkl'


# Load from file
with open(path+fnm, 'rb') as file:
    MLlvl3 = pickle.load(file)


In [6]:
lvl1_predval_tabs = {}
for mlname in MLlvl1['fold_0'].keys():
    for target in MLlvl1['fold_0'][mlname].keys():
            df_ = pd.DataFrame()
            for fold in sorted(MLlvl1.keys()):
                    #print(fold, target)
                    tab = MLlvl1[fold][mlname][target]['predval_test']
                    fold_vec = np.full(len(tab.index), fold)
                    tab.insert(0, 'fold', fold_vec)
                    df_ = pd.concat([df_, tab], axis=0)
            
            lvl1_predval_tabs[target+'__'+mlname] = df_

In [7]:
lvl2_predval_tabs = {}
for mlname in MLlvl2['fold_0'].keys():
    for target in MLlvl2['fold_0'][mlname].keys():
            df_ = pd.DataFrame()
            for fold in sorted(MLlvl2.keys()):
                    #print(fold, target)
                    tab = MLlvl2[fold][mlname][target]['predval_test']
                    fold_vec = np.full(len(tab.index), fold)
                    tab.insert(0, 'fold', fold_vec)
                    df_ = pd.concat([df_, tab], axis=0)
            
            lvl2_predval_tabs[target+'__'+mlname] = df_

In [8]:
lvl3_predval_tabs = {}
for mlname in MLlvl3['fold_0'].keys():
    for target in MLlvl3['fold_0'][mlname].keys():
            df_ = pd.DataFrame()
            for fold in sorted(MLlvl3.keys()):
                    #print(fold, target)
                    tab = MLlvl3[fold][mlname][target]['predval_test']
                    fold_vec = np.full(len(tab.index), fold)
                    tab.insert(0, 'fold', fold_vec)
                    df_ = pd.concat([df_, tab], axis=0)
            
            lvl3_predval_tabs[target+'__'+mlname] = df_

In [ ]:
for lvlname, lvldct in zip(['MLlvl1', 'MLlvl2', 'MLlvl3'],
                           [lvl1_predval_tabs, lvl2_predval_tabs, lvl3_predval_tabs]):
    for target in lvldct.keys():
        if 'MoCA__' in target or 'global_z_' in target:
            print(target)

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import os

def bootstrap_target(lvlname, target, data, path, n_trials=5000):
    if not ('MoCA__' in target or 'global_z_' in target):
        return  # Skip irrelevant targets

    dt_boot = {}

    for i in range(n_trials):
        boot = resample(data, replace=True, n_samples=len(data), random_state=i)
        metrics = {
            'r2': [],
            'mse': [],
            'mae': [],
            'cor': [],
        }
        cols = boot.columns[2:]

        for col in cols:
            y_true, y_pred = boot['y_obs'], boot[col]

            def safe_r2(y_true, y_pred):
                return np.nan if np.isnan(y_pred).any() or np.isnan(y_true).any() else r2_score(y_true, y_pred)

            def safe_mse(y_true, y_pred):
                return np.nan if np.isnan(y_pred).any() or np.isnan(y_true).any() else mean_squared_error(y_true, y_pred)

            def safe_mae(y_true, y_pred):
                return np.nan if np.isnan(y_pred).any() or np.isnan(y_true).any() else mean_absolute_error(y_true, y_pred)

            def safe_cor(y_true, y_pred):
                return (np.nan, np.nan) if np.isnan(y_pred).any() or np.isnan(y_true).any() else pearsonr(y_true, y_pred)

            r2 = safe_r2(y_true, y_pred)
            mse = safe_mse(y_true, y_pred)
            mae = safe_mae(y_true, y_pred)
            cor, _ = safe_cor(y_true, y_pred)

            metrics['cor'].append(cor)
            metrics['r2'].append(r2)
            metrics['mse'].append(mse)
            metrics['mae'].append(mae)

        dt_boot[i] = pd.DataFrame(metrics, index=cols)

    # Convert dictionary to DataFrames
    bt_r2, bt_mse, bt_mae, bt_cor = (pd.DataFrame({k: v[metric] for k, v in dt_boot.items()}).T for metric in ['r2', 'mse', 'mae', 'cor'])

    # Compute and save mean performance
    bt_mean = pd.DataFrame({
        'r2': bt_r2.mean(),
        'mse': bt_mse.mean(),
        'mae': bt_mae.mean(),
        'cor': bt_cor.mean()
    })

    # Ensure output directory exists
    outdir = os.path.join(path, 'MLout', 'performance_boot_ZnoLG_lvl1_lvl2_lvlFlat_tabs')
    os.makedirs(outdir, exist_ok=True)

    for name, df in zip(['r2', 'mse', 'mae', 'cor'], [bt_r2, bt_mse, bt_mae, bt_cor]):
        df.to_csv(f"{outdir}/{lvlname}_{target}_{name}_performance_bootstr.csv")

    bt_mean.to_csv(f"{outdir}/{lvlname}_{target}_Mean_performance_bootstr.csv")

    # Optional: return summary for later display (if you want to gather them)
    return target, lvlname, bt_mean


# Main parallel execution
from joblib import parallel_backend

with parallel_backend('loky', inner_max_num_threads=1):  # Prevents oversubscription
    for lvlname, lvldct in zip(['MLlvl1', 'MLlvl2', 'MLlvl3'],
                               [lvl1_predval_tabs, lvl2_predval_tabs, lvl3_predval_tabs]):

        results = Parallel(n_jobs=10)(
            delayed(bootstrap_target)(lvlname, target, lvldct[target], path)
            for target in lvldct.keys()
            if 'MoCA__' in target or 'global_z_' in target
        )

        # Print summary if needed
        for res in results:
            if res:
                target, lvlname, bt_mean = res
                print(target, lvlname)
                print("Mean Performance based on Bootstrapping of Predicted and Observed Values across Test Folds")
                display(bt_mean.sort_values(by='r2', ascending=False))


In [ ]:
# #bootstrapping

# from sklearn.utils import resample

# for lvlname, lvldct in zip(['MLlvl1', 'MLlvl2', 'MLlvl3'],
#                            [lvl1_predval_tabs, lvl2_predval_tabs, lvl3_predval_tabs]):
#     for target in lvldct.keys():

#         if 'MoCA__' in target or 'global_z__' in target:
#             #

#             # Number of bootstrapping trials
#             n_trials = 5000  
#             # Dictionary to store bootstrapped performance metrics
#             dt_boot = {}
            
#             # Perform bootstrapping
#             for i in range(n_trials):
#                 data = lvldct[target]
#                 boot = resample(data, replace=True, n_samples=len(data), random_state=i)
#                 # Compute performance metrics for each column (except last two)
#                 metrics = {
#                     'r2': [],
#                     'mse': [],
#                     'mae': [],
#                     'cor': [],
#                 }
#                 cols = boot.columns[2:]

#                 for col in cols:
#                     y_true, y_pred = boot['y_obs'], boot[col]
#                     #performance (test)
#                     def safe_r2(y_true, y_pred):
#                         return np.nan if np.isnan(y_pred).any() or np.isnan(y_true).any() else r2_score(y_true, y_pred)

#                     def safe_mse(y_true, y_pred):
#                         return np.nan if np.isnan(y_pred).any() or np.isnan(y_true).any() else mean_squared_error(y_true, y_pred)

#                     def safe_mae(y_true, y_pred):
#                         return np.nan if np.isnan(y_pred).any() or np.isnan(y_true).any() else mean_absolute_error(y_true, y_pred)

#                     def safe_cor(y_true, y_pred):
#                         return (np.nan, np.nan) if np.isnan(y_pred).any() or np.isnan(y_true).any() else pearsonr(y_true, y_pred)

#                     # Usage:
#                     r2 = safe_r2(y_true, y_pred)
#                     mse = safe_mse(y_true, y_pred)
#                     mae = safe_mae(y_true, y_pred)
#                     cor, pval = safe_cor(y_true, y_pred)

#                     metrics['cor'].append(cor)
#                     metrics['r2'].append(r2)
#                     metrics['mse'].append(mse)
#                     metrics['mae'].append(mae)

#                 # Store results in dictionary as DataFrame
#                 dt_boot[i] = pd.DataFrame(metrics, index=cols)

#             # Convert dictionary to DataFrames for each metric
#             bt_r2, bt_mse, bt_mae, bt_cor = (pd.DataFrame({k: v[metric] for k, v in dt_boot.items()}).T for metric in ['r2', 'mse', 'mae', 'cor'])



#             # Compute and save mean performance table
#             bt_mean = pd.DataFrame({
#                 'r2': bt_r2.mean(),
#                 'mse': bt_mse.mean(),
#                 'mae': bt_mae.mean(),
#                 'cor': bt_cor.mean()
#             })


#             # Save results
#             for name, df in zip(['r2', 'mse', 'mae', 'cor'], [bt_r2, bt_mse, bt_mae, bt_cor]):
#                 df.to_csv(f"{path}/MLout/performance_boot_lvl1_lvl2_lvlFlat_tabs/{lvlname}_8target_{target}_{name}_performance_bootstr.csv")
                                

#             bt_mean.to_csv(f"{path}/MLout/performance_boot_lvl1_lvl2_lvlFlat_tabs/{lvlname}_8target_{target}_Mean_performance_bootstr.csv")


#             # Display results
#             print(target, name, lvlname)
#             print("Mean Performance based on Bootstrapping of Predicted and Observed Values across Test Folds")
#             display(bt_mean.sort_values(by='r2', ascending=False))

